# 01 - Scientific Computing Foundation

This notebook introduces the shared validation and precision conventions used by the educational Quantum Space AGI engine.

The utilities here are research-oriented: they reject ambiguous or unsafe numeric state, but they do not represent approved flight software assumptions.

## Key conventions

- Real calculations use finite `Float64` values where NumPy arrays are created.
- Absolute and relative tolerance are combined as
$$|a-b| \leq a_{tol} + r_{tol}\max(|a|,|b|).$$
- Domain-specific models may require different tolerances.
- Structured `ScientificError` exceptions distinguish invalid dimensions, non-finite values, and other scientific failures.

**Expected result:** scalar comparisons should accept clearly close values and reject values outside the configured threshold.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd()
repo_root = next((p for p in (cwd, *cwd.parents) if (p / "src").is_dir()), cwd)
sys.path.insert(0, str(repo_root / "src"))

from quantum_space_agi.scientific.errors import ScientificError
from quantum_space_agi.scientific.precision import (
    ComparisonTolerance,
    is_close,
)
from quantum_space_agi.scientific.validation import validate_finite, validate_vector


## Finite scalar validation

A research calculation should fail clearly when a component is not a finite real number. This avoids silently propagating `NaN` or infinity into mission-design results.

In [2]:
print(validate_finite(9.81, field="gravity"))

try:
    validate_finite(float("inf"), field="gravity")
except ScientificError as exc:
    print(exc.code.value, str(exc))


9.81
non_finite_value non_finite_value: Value must be finite. (field=gravity; expected='a finite number'; received=inf)


**Expected output:** `9.81`, followed by `non_finite_value` and a descriptive message.

## Supported vector shapes and precision

The foundation deliberately supports 2D and 3D numeric vectors. It does not attach units or coordinate-frame metadata; callers must manage those assumptions explicitly.

In [3]:
print(validate_vector([1.0, 2.0]))
print(validate_vector([3.0, 4.0, 5.0]))

default_close = is_close(1.000000001, 1.0)
strict_close = is_close(
    1.000000001,
    1.0,
    tolerance=ComparisonTolerance(atol=1e-12, rtol=1e-12),
)

print("default tolerance:", default_close)
print("1e-12 tolerance:", strict_close)


(1.0, 2.0)
(3.0, 4.0, 5.0)
default tolerance: True
1e-12 tolerance: False


**Validation expectation:**

- The default comparison should return `True` under the provisional `1e-12` absolute and `1e-9` relative defaults.
- The stricter comparison should return `True` for `atol=rtol=1e-12`? Here the difference is approximately `1e-9`, so it should be `False`; this demonstrates the importance of choosing tolerances by context.

## Numerical limitations

- Floating-point rounding is not eliminated by validation.
- Relative comparisons become less meaningful near values affected by large cancellation or measurement uncertainty.
- `ComparisonTolerance` defaults are provisional ARC-04 policies, not universal aerospace standards.